In [6]:
import time
import re  
from urllib.parse import quote
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from tqdm import tqdm
from newspaper import Article, Config 

def preprocess_sentence_kr(w):
    w = w.strip()
    w = re.sub(r"[^0-9가-힣?.!,¿]+", " ", w) 
    w = w.strip() 
    return w

def crawl_naver_news_to_txt(keyword, date_option='1y', file_path='./naver_news.txt',count=75):
    print(f"[{keyword}] 뉴스 데이터 크롤링을 시작합니다...")
    
    encoded_keyword = quote(keyword)
    url = f'https://search.naver.com/search.naver?ssc=tab.news.all&query={encoded_keyword}&sm=tab_opt&sort=0&photo=0&field=0&pd=0&ds=&de=&docid=&related=0&mynews=0&office_type=0&office_section_code=0&news_office_checked=&nso=so%3Ar%2Cp%3A{date_option}&is_sug_officeid=0&office_category=0&service_area=0' 
    
    driver = webdriver.Chrome()
    driver.get(url)
    time.sleep(2)
    
    body = driver.find_element(By.TAG_NAME, 'body')
    for i in range(count):
        body.send_keys(Keys.END)
        time.sleep(0.75)
        
    # 마스터께서 작성하신 원래 CSS 선택자로 복구하였습니다.
    aTag = driver.find_elements(By.CSS_SELECTOR, '.fender-ui_228e3bd1.XEtVZ4N7DI2Pv29xe7f3')
    href_list = []
    for i in aTag:
        href = i.get_attribute('href')
        if href:
            href_list.append(href)
            
    # ✨ [추가된 핵심 로직] 중복 링크 제거 (기존 정렬 순서 유지)
    original_count = len(href_list)
    href_list = list(dict.fromkeys(href_list))
    print(f"-> 🧹 중복 링크 제거 완료: 총 {original_count}개 중 {original_count - len(href_list)}개의 중복을 삭제하고 {len(href_list)}개의 링크를 최종 확보했습니다.")
    
    # 봇 차단을 피하기 위한 User-Agent 설정
    config = Config()
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    config.request_timeout = 10

    f = open(file_path, 'w', encoding='utf-8')
    f.write("url||content\n")
    
    for i in tqdm(range(len(href_list))):
        link = href_list[i]
        try:
            article = Article(link, language='ko', config=config)
            article.download()
            article.parse()
            
            content = article.text
            
            # 봇 차단 등으로 내용이 비어있으면 다음 기사로 넘어갑니다.
            if not content.strip():
                continue
                
            clean_content = preprocess_sentence_kr(content)
            
            if clean_content:
                f.write(f"{link}||{clean_content}\n")
                
        except Exception as e:
            continue
            
    f.close()
    driver.quit()
    print(f"크롤링 완료! 데이터가 '{file_path}'에 안전하게 저장되었습니다.")

In [7]:
crawl_naver_news_to_txt('육아용품대여', date_option='1y', file_path='./data/육아용품대여.txt',count=100)

[육아용품대여] 뉴스 데이터 크롤링을 시작합니다...
-> 🧹 중복 링크 제거 완료: 총 420개 중 138개의 중복을 삭제하고 282개의 링크를 최종 확보했습니다.


100%|██████████| 282/282 [05:17<00:00,  1.13s/it]


크롤링 완료! 데이터가 './data/육아용품대여.txt'에 안전하게 저장되었습니다.


In [8]:
crawl_naver_news_to_txt('육아용품사용기간', date_option='1y', file_path='./data/육아용품사용기간.txt',count=100)

[육아용품사용기간] 뉴스 데이터 크롤링을 시작합니다...
-> 🧹 중복 링크 제거 완료: 총 1010개 중 248개의 중복을 삭제하고 762개의 링크를 최종 확보했습니다.


100%|██████████| 762/762 [15:02<00:00,  1.18s/it]


크롤링 완료! 데이터가 './data/육아용품사용기간.txt'에 안전하게 저장되었습니다.


In [9]:
crawl_naver_news_to_txt('육아용품구독기간', date_option='1y', file_path='./data/육아용품구독기간.txt',count=100)

[육아용품구독기간] 뉴스 데이터 크롤링을 시작합니다...
-> 🧹 중복 링크 제거 완료: 총 790개 중 120개의 중복을 삭제하고 670개의 링크를 최종 확보했습니다.


100%|██████████| 670/670 [16:33<00:00,  1.48s/it] 


크롤링 완료! 데이터가 './data/육아용품구독기간.txt'에 안전하게 저장되었습니다.
